# 04: Optuna tuning (EXP-003)

Tunes Logistic Regression, XGBoost and Random Forest with Optuna and logs everything to MLflow: one parent run per model, one nested run per trial.

**Method (no test leakage):** the search maximises 5-fold CV F1 on the *training* split, using the same folds as the baselines. Trial 0 is the baseline configuration (except Random Forest `max_depth`, whose baseline is unlimited and not expressible in the space), so for Logistic Regression and XGBoost the tuned CV score cannot fall below the baseline. The test split is scored once per model after the search.

**Expect:** the best CV score is optimistically biased (it is the maximum of many noisy scores). Judge by test F1 and by whether the gain is larger than the fold spread (about 0.015 to 0.025).

**Runtime:** about 40 trials per model; Random Forest and XGBoost take the longest (roughly 10 to 25 minutes in total on Colab CPU). Only one session should write to the MLflow store at a time.

In [ ]:
# --- AI Experiment Lab: standard Colab setup cell ---------------------------------
import os, subprocess, sys

REPO_URL = "https://github.com/sudhanshu230230/AI_Experiment_Lab.git"   # <-- edit
REPO_DIR = "/content/ai-experiment-lab"
DRIVE_ROOT = "/content/drive/MyDrive/ai-experiment-lab"
EXTRAS = "ml,tracking"          # pip extras from pyproject.toml

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/mlflow", exist_ok=True)

# For a private repo, store a token in Colab Secrets as GITHUB_TOKEN (never paste it here).
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# An editable install adds a .pth file, which a running kernel does not re-read.
# Put src/ on sys.path so the import works in this session without a restart.
src_dir = f"{REPO_DIR}/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Tracking store: SQLite database on Drive (accepted in ADR-001; the file: store is deprecated).
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{DRIVE_ROOT}/mlflow/mlflow.db"
os.environ["AEL_DATA_DIR"] = f"{DRIVE_ROOT}/data"

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code version (commit):", commit)
print("Data dir:", os.environ["AEL_DATA_DIR"])

import ai_experiment_lab
print("ai_experiment_lab", ai_experiment_lab.__version__, "imported OK")

In [ ]:
from pathlib import Path
import pandas as pd

from ai_experiment_lab.data import load_csv, validate_dataframe, validation_thresholds
from ai_experiment_lab.preprocessing import split_data
from ai_experiment_lab.training import run_baselines
from ai_experiment_lab.tuning import tune_model
from ai_experiment_lab.tracking import (
    backup_tracking_db, compare_runs, log_tuning_result, setup_tracking,
)
from ai_experiment_lab.utils.config import get_data_dir, load_config

cfg = load_config(f"{REPO_DIR}/configs/default.yaml")
models_cfg = load_config(f"{REPO_DIR}/configs/model_configs.yaml")["classification"]
tcfg = load_config(f"{REPO_DIR}/configs/tuning.yaml")["tuning"]
d, sp, seed = cfg["dataset"], cfg["split"], cfg["project"]["random_seed"]

df, info = load_csv(Path(get_data_dir()) / d["file"])
validate_dataframe(df, d["target"], d["task"], validation_thresholds(cfg), d["id_columns"]).raise_if_rejected()
split = split_data(df, d["target"], d["positive_label"], d["id_columns"],
                   sp["test_size"], seed, sp["stratify"])
print("train:", split.X_train.shape, " test:", split.X_test.shape)
print("trials per model:", tcfg["n_trials"])

## Baselines (for comparison; same numbers as EXP-001/002)

In [ ]:
CV_FOLDS = 5
base = run_baselines(split, models_cfg, cfg["preprocessing"], seed=seed, cv_folds=CV_FOLDS)
base_tbl = base.table.set_index("model")
print(base_tbl[["cv_f1", "test_f1", "test_recall", "test_roc_auc"]].round(4).to_string())

## Tune each model and log to MLflow

In [ ]:
import time
setup_tracking(cfg["tracking"]["experiment_name"])
results, run_ids = {}, {}
for name in models_cfg:
    t0 = time.perf_counter()
    res = tune_model(
        name, split, tcfg["spaces"][name], models_cfg[name].get("params", {}),
        cfg["preprocessing"], seed=seed, cv_folds=CV_FOLDS, n_trials=tcfg["n_trials"],
        timeout_seconds=tcfg.get("timeout_seconds"),
        startup_trials=tcfg["pruner_startup_trials"], warmup_folds=tcfg["pruner_warmup_folds"],
    )
    results[name] = res
    run_ids[name] = log_tuning_result(
        res, split, info, cfg, cv_folds=CV_FOLDS, n_trials_requested=tcfg["n_trials"],
        extra_tags={"experiment.id": "EXP-003", "notebook": "04_optuna_tuning"})
    print(f"{name:20s} done in {time.perf_counter()-t0:6.1f}s  "
          f"complete={res.n_complete} pruned={res.n_pruned}  best cv_f1={res.best_cv_f1:.4f}")

## Baseline vs tuned

In [ ]:
rows = []
for name, res in results.items():
    b = base_tbl.loc[name]
    rows.append({
        "model": name,
        "base_cv_f1": b["cv_f1"], "tuned_cv_f1": res.best_cv_f1,
        "cv_gain": res.best_cv_f1 - b["cv_f1"],
        "tuned_cv_std": pd.Series(res.cv_fold_scores).std(ddof=0),
        "base_test_f1": b["test_f1"], "tuned_test_f1": res.test_metrics["f1"],
        "test_gain": res.test_metrics["f1"] - b["test_f1"],
        "tuned_test_recall": res.test_metrics["recall"],
        "tuned_test_precision": res.test_metrics["precision"],
        "tuned_test_roc_auc": res.test_metrics["roc_auc"],
        "tuned_test_pr_auc": res.test_metrics["pr_auc"],
    })
cmp = pd.DataFrame(rows).sort_values("tuned_cv_f1", ascending=False).reset_index(drop=True)
pd.set_option("display.width", 250)
print(cmp.round(4).to_string(index=False))

## Best parameters and search behaviour

In [ ]:
for name, res in results.items():
    print(f"\n== {name} ==  best trial cv_f1={res.best_cv_f1:.4f}  search {res.seconds:.0f}s")
    print("best params:", {k: (round(v, 5) if isinstance(v, float) else v) for k, v in res.best_params.items()})
    top = res.trials[res.trials.state == "COMPLETE"].sort_values("cv_f1", ascending=False).head(5)
    print(top.round(4).to_string(index=False))

## Search curve: best CV F1 so far, per model

A curve that is flat from early on means the search space is not paying off; a curve still rising at the last trial means more trials could help.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.5))
for name, res in results.items():
    t = res.trials.sort_values("number")
    ax.plot(t["number"], t["cv_f1"].cummax(), label=name)
    ax.axhline(base_tbl.loc[name, "cv_f1"], ls=":", lw=0.8, color=ax.lines[-1].get_color())
ax.set_xlabel("trial"); ax.set_ylabel("best CV F1 so far"); ax.legend(); plt.tight_layout(); plt.show()

## Stored runs

In [ ]:
t = compare_runs(cfg["tracking"]["experiment_name"], order_by="cv_f1")
t[["model", "stage", "cv_f1", "cv_f1_std", "test_f1", "test_recall", "test_roc_auc", "code_version"]].round(4)

In [ ]:
print("backup:", backup_tracking_db())

## Record

Paste: the **Baseline vs tuned** table, the **best parameters** output, the stored-runs table, and the backup path. Results go in `docs/experiments/EXP-003-optuna-tuning.md`.